# 01 - Data Loading & Schema Profiling

**Goal**: Load all raw CSV tables, profile every column (dtype, null rate, unique count, samples),
and document the complete data dictionary.

**Learning concepts**: Pandas data loading, dtype specification, memory management for large CSVs,
understanding relational data schemas.

---

In [ ]:
import pandas as pd
import numpy as np
from talentlens.config import (
    POSTINGS_CSV, COMPANIES_CSV, COMPANY_INDUSTRIES_CSV,
    COMPANY_SPECIALITIES_CSV, EMPLOYEE_COUNTS_CSV,
    BENEFITS_CSV, JOB_INDUSTRIES_CSV, JOB_SKILLS_CSV,
    SALARIES_CSV, INDUSTRIES_MAP_CSV, SKILLS_MAP_CSV,
)
from talentlens.dataset import load_raw_postings, load_secondary_tables

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 1. Load the Main Postings Table

The main dataset has **3.38M rows** and is ~493MB. We load it with specified dtypes
to avoid pandas guessing wrong types (which wastes memory).

> **Tip**: For development, use `nrows=10000` to load a small sample first.
> Once your code works, remove the limit to process the full dataset.

In [ ]:
# Load a small sample first to understand the schema
# Change nrows=None to load the full dataset when ready
df_postings = load_raw_postings(nrows=10000)
print(f"Shape: {df_postings.shape}")
print(f"Memory usage: {df_postings.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df_postings.head(3)

## 2. Profile the Postings Columns

For each column, we want to know:
- **dtype**: What type did pandas assign?
- **null rate**: What percentage of values are missing?
- **unique count**: How many distinct values?
- **sample values**: What does the data actually look like?

In [ ]:
def profile_dataframe(df: pd.DataFrame, name: str = "DataFrame") -> pd.DataFrame:
    """Generate a profiling summary for every column in a DataFrame."""
    profile = pd.DataFrame({
        'dtype': df.dtypes,
        'non_null': df.count(),
        'null_count': df.isna().sum(),
        'null_pct': (df.isna().sum() / len(df) * 100).round(1),
        'unique': df.nunique(),
        'sample_values': [df[col].dropna().head(3).tolist() for col in df.columns],
    })
    print(f"\n{'='*60}")
    print(f"  {name}: {len(df):,} rows x {len(df.columns)} columns")
    print(f"{'='*60}")
    return profile

In [ ]:
profile_postings = profile_dataframe(df_postings, "Postings")
profile_postings

## 3. Load & Profile All Secondary Tables

The dataset is **relational** — like a database with multiple linked tables:

```
postings (main) ──┬── job_skills ──── skills (mapping)
                   ├── job_industries ── industries (mapping)
                   ├── benefits
                   ├── salaries
                   └── companies ──┬── company_industries
                                   ├── company_specialities
                                   └── employee_counts
```

Tables are linked via `job_id` or `company_id` foreign keys.

In [ ]:
tables = load_secondary_tables()

for name, df in tables.items():
    display(profile_dataframe(df, name))

## 4. Explore the Mapping Tables

These are small lookup tables that decode IDs into human-readable names.

In [ ]:
# Skills mapping: 35 skill categories
print("=== Skills Mapping ===")
display(tables['skills_map'])

print(f"\n=== Industries Mapping (first 20 of {len(tables['industries_map'])}) ===")
display(tables['industries_map'].head(20))

## 5. Join job_skills with Skills Names

The `job_skills` table uses abbreviations like `IT`, `SALE`, `DSGN`.
Let's join with the mapping table to see the full skill names.

In [ ]:
# Join skills abbreviations to full names
skills_enriched = tables['job_skills'].merge(
    tables['skills_map'],
    on='skill_abr',
    how='left'
)
print(f"Job-skill associations: {len(skills_enriched):,}")
print(f"\nTop 15 skills by frequency:")
skills_enriched['skill_name'].value_counts().head(15)

## 6. Quick Stats Summary

Before moving to cleaning, let's capture the key numbers.

In [ ]:
# Load full postings count without loading all data into memory
# (just count the lines)
import subprocess
result = subprocess.run(
    ['wc', '-l', str(POSTINGS_CSV)],
    capture_output=True, text=True
)
total_lines = result.stdout.strip().split()[0] if result.returncode == 0 else "unknown"

print("=== Dataset Summary ===")
print(f"Total postings (raw CSV lines): {total_lines}")
print(f"Sample loaded: {len(df_postings):,} rows")
print(f"Columns: {df_postings.shape[1]}")
print(f"\nSecondary tables:")
for name, df in tables.items():
    print(f"  {name}: {len(df):,} rows x {df.shape[1]} cols")

print(f"\nKey null rates in postings (sample):")
key_cols = ['description', 'title', 'med_salary', 'remote_allowed', 
            'formatted_experience_level', 'skills_desc', 'applies', 'views']
for col in key_cols:
    if col in df_postings.columns:
        null_pct = df_postings[col].isna().mean() * 100
        print(f"  {col}: {null_pct:.1f}% null")

## Key Observations

*(Fill in after running the notebook)*

1. **Postings table**: X rows, Y columns. Description null rate = Z%.
2. **Salary coverage**: Only X% of postings have salary data.
3. **Skills coverage**: X job-skill associations across Y unique skills.
4. **Next step**: Clean the data in notebook 02.

---

**→ Next notebook**: `02-mp-data-cleaning.ipynb`